# Tomato Irrigation - Isolation Forest 

This notebook is a grounded, minimal, and **readable** pipeline for anomaly detection on the Parma tomato dataset (lines **L100 / L60 / L30**).  
It keeps what is necessary to: **build time‑aware features, train Isolation Forest on post‑24 July (stable period), detect anomalies, group them into periods, visualize, and compute a simple precision metric vs. **pre‑declared** anomaly windows.**


---

### Design choices 
- **Training window**: only data **>= 2023‑07‑25 00:00** (sensor repositioning done; data stable per authors).  
- **Model**: `IsolationForest` threshold = a **quantile of training scores** (e.g. 1%).  
- **Anomaly grouping**: merge adjacent anomaly points (10‑min cadence) with up to 20 minutes gap into one **period**.  
- **Precision**: (a) **point‑wise precision** and (b) **period‑wise precision** against **pre‑declared** windows (GT - Ground Truth windows).  
- **Watering shading**: inferred from **positive deltas** in `current_volume` per line (drip irrigation).  
- **Optional**: SHAP explanations for isolation trees (added plain english explanation for maximum simplicity).



In [3]:
import pandas as pd
import numpy as np
import sklearn
import plotly
import shap
import matplotlib 

try:
    print("shap:", shap.__version__)
except ImportError:
    print("shap: not installed")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("plotly:", plotly.__version__)
print("matplotlib:", matplotlib.__version__)

shap: 0.48.0
pandas: 2.3.0
numpy: 1.26.4
scikit-learn: 1.7.0
plotly: 6.3.0
matplotlib: 3.10.3


In [4]:

import os, numpy as np, pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots




DATA_PATH = Path('long_format_data.csv')  # <- change if needed
TRAIN_START = pd.Timestamp('2023-07-25 00:00:00')    # post-repositioning stable window
TIMESTEP_MIN = 10                                    # dataset cadence (minutes)
LINE_CHOICES = ['L100','L60','L30']
DEFAULT_LINE = 'L30'

# anomaly threshold = lower q-quantile of **training** scores (decision_function)
ANOMALY_SCORE_QUANTILE = 0.01   # 1% is a reasonable starting point

# anomaly grouping: points <= this gap (minutes) are merged into one period
GROUP_GAP_MIN = 20

# watering detection: minimal increase in water meter to consider "watering"
WATER_DELTA_MIN = 0.001  # m3, tiny positive delta

ENABLE_SHAP = True      # set False to skip fast
SHAP_TOP_K = 3



In [5]:
import pandas as pd


def read_data(path: Path) -> pd.DataFrame:
    """Load long-format CSV and cast types."""
    df = pd.read_csv(path)
    df['dt'] = pd.to_datetime(df['dt'])
    # ensure line is categorical with expected order
    df['line'] = pd.Categorical(df['line'], categories=['L100','L60','L30'], ordered=True)
    return df.sort_values(['line','dt']).reset_index(drop=True)


def add_watering_delta(df: pd.DataFrame) -> pd.DataFrame:
    """Compute per-line water delta from cumulative `current_volume` and a boolean flag `watering`.
    Assumes `current_volume` is cumulative per line.
    """
    df = df.copy()
    df['water_delta'] = df.groupby('line')['current_volume'].diff().fillna(0.0)
    # negative resets -> set to 0
    df.loc[df['water_delta'] < 0, 'water_delta'] = 0.0
    df['watering'] = df['water_delta'] > WATER_DELTA_MIN
    return df


def build_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create minimal, time-aware features.
    We use only a compact set to keep the model simple and robust.

    Base features: soil: humidity, temperature, electrical_conductivity
                   
    Time features: rolling mean/std for soil humidity, temperature & electrical_conductivity
                   at 3h and 24h, 1h delta (difference) and 6h slope approx (difference / hours)
    """
    df = df.copy()
    df = df.sort_values(['line','dt'])
    feat_cols = ['humidity', 'temperature', 'electrical_conductivity']

    # windows in number of rows (10-min cadence)
    w3h  = int(3*60 / TIMESTEP_MIN)   # 18
    w24h = int(24*60 / TIMESTEP_MIN)  # 144
    w1h  = int(60 / TIMESTEP_MIN)     # 6
    w6h  = int(6*60 / TIMESTEP_MIN)   # 36

    def _per_line(g: pd.DataFrame) -> pd.DataFrame:
        g = g.copy()
        # rolling stats for soil metrics
        g['hum_roll3h']   = g['humidity'].rolling(w3h,  min_periods=1).mean()
        g['hum_std3h']    = g['humidity'].rolling(w3h,  min_periods=1).std().fillna(0)
        g['hum_roll24h']  = g['humidity'].rolling(w24h, min_periods=1).mean()

        g['temp_roll3h']  = g['temperature'].rolling(w3h,  min_periods=1).mean()
        g['temp_std3h']   = g['temperature'].rolling(w3h,  min_periods=1).std().fillna(0)
        g['temp_roll24h'] = g['temperature'].rolling(w24h, min_periods=1).mean()

        # electrical_conductivity rolling stats
        g['ec_roll3h']    = g['electrical_conductivity'].rolling(w3h,  min_periods=1).mean()
        g['ec_std3h']     = g['electrical_conductivity'].rolling(w3h,  min_periods=1).std().fillna(0)
        g['ec_roll24h']   = g['electrical_conductivity'].rolling(w24h, min_periods=1).mean()

        # deltas / slopes
        g['hum_delta_1h']   = g['humidity'] - g['humidity'].shift(w1h)
        g['temp_delta_1h']  = g['temperature'] - g['temperature'].shift(w1h)
        g['ec_delta_1h']    = g['electrical_conductivity'] - g['electrical_conductivity'].shift(w1h)

        g['hum_slope_6h']   = (g['humidity'] - g['humidity'].shift(w6h)) / 6.0
        g['temp_slope_6h']  = (g['temperature'] - g['temperature'].shift(w6h)) / 6.0
        g['ec_slope_6h']    = (g['electrical_conductivity'] - g['electrical_conductivity'].shift(w6h)) / 6.0

        return g

    df = df.groupby('line', group_keys=False).apply(_per_line)

    # final feature list (include EC derived features)
    X_cols = [
        'humidity','temperature','electrical_conductivity',
        'hum_roll3h','hum_std3h','hum_roll24h',
        'temp_roll3h','temp_std3h','temp_roll24h',
        'ec_roll3h','ec_std3h','ec_roll24h',
        'hum_delta_1h','temp_delta_1h','ec_delta_1h',
        'hum_slope_6h','temp_slope_6h','ec_slope_6h'
    ]

    return df, X_cols


def train_iforest(df_line: pd.DataFrame, X_cols: list, train_start: pd.Timestamp):
    """Fit IsolationForest on post-`train_start` subset and return model, scaler, threshold, and scores.
    - Scores are sklearn `decision_function` (higher is more normal).
    - Threshold is the q-quantile of **training** scores (lower tail -> anomalies).
    """
    d = df_line.copy()
    train_mask = d['dt'] >= train_start
    print(train_mask.isna().sum())
    
    X_train = d.loc[train_mask, X_cols].to_numpy()
    X_all   = d[X_cols].to_numpy()

    # simple scaling helps numerical stability
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_all_s   = scaler.transform(X_all)

    print("NaNs in X_train:", np.isnan(X_train).sum())
    print("NaNs in X_all:", np.isnan(X_all).sum())
    
    iforest = IsolationForest(
        n_estimators=800,
        max_samples=512,
        max_features=0.8,
        bootstrap=True,
        contamination='auto',
        random_state=42,
        n_jobs=-1
    )
    iforest.fit(X_train_s)
    # decision_function: positive ~ normal, negative ~ anomalous
    scores_all = iforest.decision_function(X_all_s)
    thr = np.quantile(scores_all[train_mask.values], ANOMALY_SCORE_QUANTILE)
    return iforest, scaler, thr, scores_all


def label_and_group_anomalies(df_line: pd.DataFrame, scores_all: np.ndarray, thr: float):
    """Add `score` and `is_anom` columns and group contiguous anomalies into periods.
    Returns: df_line (augmented), and a Periods DataFrame with columns: [start, end, n_points].
    """
    d = df_line.copy()
    d['score'] = scores_all
    d['is_anom'] = d['score'] < thr

    # group anomalies into periods — contiguous if gap <= GROUP_GAP_MIN
    periods = []
    curr_start = None
    prev_time = None
    gap = pd.Timedelta(minutes=GROUP_GAP_MIN)

    for row in d.loc[d['is_anom']].itertuples():
        t = row.dt
        if (curr_start is None) or (prev_time is None) or (t - prev_time > gap):
            # start new
            curr_start = t
            prev_time = t
            periods.append([curr_start, curr_start, 1])
        else:
            # extend
            periods[-1][1] = t
            periods[-1][2] += 1
            prev_time = t

    periods_df = pd.DataFrame(periods, columns=['start','end','n_points'])
    return d, periods_df


def precision_against_windows(periods_df: pd.DataFrame, df_points: pd.DataFrame, gt_windows: list):
    """Compute a simple precision vs. **predeclared** ground-truth windows.
    - gt_windows: list of (start_ts, end_ts, label) tuples (edit below).
    Returns a dict with:
        - point_precision: fraction of predicted anomaly *points* that fall in any GT window
        - period_precision: fraction of predicted *periods* that intersect any GT window
    """
    if len(gt_windows) == 0:
        return {'point_precision': np.nan, 'period_precision': np.nan}

    # prepare boolean mask for points
    mask = np.zeros(len(df_points), dtype=bool)
    for (s,e,_) in gt_windows:
        mask |= (df_points['dt'] >= s) & (df_points['dt'] <= e)

    pts = df_points['is_anom'].sum()
    pts_in = (df_points['is_anom'] & mask).sum()
    point_prec = float(pts_in) / float(pts) if pts > 0 else np.nan

    # period-wise
    def _overlaps(a_start, a_end, b_start, b_end):
        return (a_start <= b_end) and (b_start <= a_end)

    hit = 0
    for row in periods_df.itertuples(index=False):
        any_overlap = any(_overlaps(row.start, row.end, s, e) for (s,e,_) in gt_windows)
        hit += int(any_overlap)
    period_prec = float(hit) / float(len(periods_df)) if len(periods_df) else np.nan

    return {'point_precision': point_prec, 'period_precision': period_prec}






def assign_gt_labels_points(df_points, gt_windows):
    labels = np.array([''] * len(df_points), dtype=object)
    for (s,e,label) in gt_windows:
        sel = (df_points['dt'] >= s) & (df_points['dt'] <= e)
        labels[sel] = np.where(labels[sel] == '', label, labels[sel] + ' | ' + label)
    out = df_points.copy()
    out['gt_window_label'] = labels
    return out

def assign_gt_labels_periods(periods_df, gt_windows):
    lab = []
    for row in periods_df.itertuples(index=False):
        labels = []
        for (s,e,label) in gt_windows:
            if (row.start <= e) and (s <= row.end):
                labels.append(label)
        lab.append(' | '.join(sorted(set(labels))) if labels else '')
    out = periods_df.copy()
    out['gt_window_label'] = lab
    return out







# === add near your utils ===

def add_shap_reasons(iforest, scaler, d_line, periods, X_cols, top_k=3):
    """
    Adds to d_line:
      shap_top1, shap_top2, shap_top3, shap_summary (only for anomalous rows)
    Adds to periods:
      shap_period_summary (mean abs SHAP across anomalies inside the period)
    """
    try:
        import shap
    except Exception as e:
        print("SHAP unavailable, skipping:", e)
        # create empty cols so downstream code still works
        for c in ['shap_top1','shap_top2','shap_top3','shap_summary']:
            if c not in d_line.columns: d_line[c] = ''
        if 'shap_period_summary' not in periods.columns:
            periods['shap_period_summary'] = ''
        return d_line, periods

    anom_mask = d_line['is_anom'].values
    if not anom_mask.any():
        for c in ['shap_top1','shap_top2','shap_top3','shap_summary']:
            if c not in d_line.columns: d_line[c] = ''
        if 'shap_period_summary' not in periods.columns:
            periods['shap_period_summary'] = ''
        return d_line, periods

    X_all = d_line[X_cols].to_numpy()
    X_all_s = scaler.transform(X_all)

    explainer   = shap.TreeExplainer(iforest)
    shap_values = explainer.shap_values(X_all_s[anom_mask])
    sv = pd.DataFrame(shap_values, columns=X_cols, index=d_line.index[anom_mask])

    top1, top2, top3, summary = [], [], [], []
    for _, row in sv.iterrows():
        order  = row.abs().sort_values(ascending=False)
        feats  = list(order.index[:top_k])
        signed = [f"{f} ({'+' if row[f]>=0 else '−'}{abs(row[f]):.3g})" for f in feats]
        while len(signed) < top_k: signed.append('')
        top1.append(signed[0]); top2.append(signed[1]); top3.append(signed[2])
        summary.append(', '.join([s for s in signed if s]))

    for c in ['shap_top1','shap_top2','shap_top3','shap_summary']:
        if c not in d_line.columns: d_line[c] = ''
    d_line.loc[d_line['is_anom'], 'shap_top1']    = top1
    d_line.loc[d_line['is_anom'], 'shap_top2']    = top2
    d_line.loc[d_line['is_anom'], 'shap_top3']    = top3
    d_line.loc[d_line['is_anom'], 'shap_summary'] = summary

    # period-level mean abs SHAP among anomalies in each period
    period_summ = []
    for rowp in periods.itertuples(index=False):
        idx = (d_line['dt'] >= rowp.start) & (d_line['dt'] <= rowp.end) & (d_line['is_anom'])
        pts = sv.loc[idx[idx].index] if idx.any() else pd.DataFrame(columns=X_cols)
        if len(pts):
            mean_abs = pts.abs().mean().sort_values(ascending=False)
            feats    = list(mean_abs.index[:top_k])
            period_summ.append(', '.join(
                f"{feat} (~{mean_abs[feat]:.3g})" for feat in feats
            ))
        else:
            period_summ.append('')
    periods['shap_period_summary'] = period_summ
    return d_line, periods



def periods_from_flag(d: pd.DataFrame, flag_col='watering', time_col='dt'):
    """
    Turn a boolean flag column into contiguous [start, end] periods.
    """
    g = d[[time_col, flag_col]].copy()
    g['flag'] = g[flag_col].astype(int)
    # new block whenever flag changes True/False
    g['block'] = (g['flag'].ne(g['flag'].shift())).cumsum()
    blocks = g[g['flag'] == 1].groupby('block')[time_col].agg(['min','max'])
    return blocks.rename(columns={'min':'start','max':'end'}).reset_index(drop=True)









# Pretty names for tooltips
PRETTY = {
    'humidity': 'soil humidity',
    'temperature': 'soil temperature',
    'electrical_conductivity': 'soil EC',
    'hum_roll3h': 'short-term humidity (3h avg)',
    'hum_std3h': 'humidity instability (3h stdev)',
    'hum_roll24h': 'daily humidity (24h avg)',
    'temp_roll3h': 'short-term temperature (3h avg)',
    'temp_std3h': 'temperature instability (3h stdev)',
    'temp_roll24h': 'daily temperature (24h avg)',
    'ec_roll3h': 'short-term EC (3h avg)',
    'ec_std3h': 'EC instability (3h stdev)',
    'ec_roll24h': 'daily EC (24h avg)',
    'hum_delta_1h': 'humidity change (1h)',
    'temp_delta_1h': 'temperature change (1h)',
    'ec_delta_1h': 'EC change (1h)',
    'hum_slope_6h': 'humidity trend (6h)',
    'temp_slope_6h': 'temperature trend (6h)',
    'ec_slope_6h': 'EC trend (6h)',
}

BASIC_VARS = [
    'humidity','temperature','electrical_conductivity',
    'hum_roll3h','hum_std3h','hum_roll24h',
    'temp_roll3h','temp_std3h','temp_roll24h',
    'ec_roll3h','ec_std3h','ec_roll24h',
    'hum_delta_1h','temp_delta_1h','ec_delta_1h',
    'hum_slope_6h','temp_slope_6h','ec_slope_6h'
]

def _build_ref_stats(df_train: pd.DataFrame, cols: list):
    """Reference stats from the TRAIN window for plain-English judgments."""
    train = df_train[cols].copy()
    ref = {
        'mean':    train.mean(),
        'std':     train.std().replace(0, 1e-9),
        'q10':     train.quantile(0.10),
        'q90':     train.quantile(0.90),
        'q90_abs': train.abs().quantile(0.90),  # for abs(delta/slope) thresholds
    }
    return ref

def _phrase_for_feature(row: pd.Series, feat: str, ref: dict) -> str:
    """Turn a single feature value into a simple phrase."""
    if not feat:
        return ''
    val = row.get(feat, np.nan)
    if pd.isna(val):
        return f"{PRETTY.get(feat, feat)} missing"

    def _hi_lo(v, mean, std, hi=1.2, lo=-1.2):
        z = (v - mean) / std
        if z >= hi:  return "high"
        if z <= lo:  return "low"
        return "unusual"

    # Base metrics (z-score vs TRAIN ref)
    if feat in ['humidity','temperature','electrical_conductivity']:
        tag = _hi_lo(val, ref['mean'].get(feat, val), ref['std'].get(feat, 1.0))
        return f"{PRETTY.get(feat, feat)} {tag}"

    # Rolling avgs vs daily baseline
    if feat == 'hum_roll3h':
        diff = row['hum_roll3h'] - row['hum_roll24h']
        band = 0.3 * ref['std'].get('humidity', 1.0)
        if diff >  band: return "short-term humidity above daily baseline"
        if diff < -band: return "short-term humidity below daily baseline"
        return "short-term humidity off baseline"

    if feat == 'temp_roll3h':
        diff = row['temp_roll3h'] - row['temp_roll24h']
        band = 0.3 * ref['std'].get('temperature', 1.0)
        if diff >  band: return "short-term temperature above daily baseline"
        if diff < -band: return "short-term temperature below daily baseline"
        return "short-term temperature off baseline"

    if feat == 'ec_roll3h':
        diff = row['ec_roll3h'] - row.get('ec_roll24h', np.nan)
        band = 0.3 * ref['std'].get('electrical_conductivity', 1.0)
        if pd.isna(diff):
            return "short-term EC off baseline"
        if diff >  band: return "short-term EC above daily baseline"
        if diff < -band: return "short-term EC below daily baseline"
        return "short-term EC off baseline"

    # Instability (compare to TRAIN 90th percentile)
    if feat == 'hum_std3h':
        return "humidity very unstable (3h)" if val > ref['q90'].get('hum_std3h', val) else "humidity stability unusual (3h)"
    if feat == 'temp_std3h':
        return "temperature very unstable (3h)" if val > ref['q90'].get('temp_std3h', val) else "temperature stability unusual (3h)"
    if feat == 'ec_std3h':
        return "EC very unstable (3h)" if val > ref['q90'].get('ec_std3h', val) else "EC stability unusual (3h)"

    # Short deltas (directional using TRAIN quantiles)
    if feat == 'hum_delta_1h':
        hi = ref['q90'].get('hum_delta_1h', val); lo = ref['q10'].get('hum_delta_1h', val)
        if val >= hi: return "sharp rise in soil humidity (1h)"
        if val <= lo: return "sharp drop in soil humidity (1h)"
        return "unusual humidity change (1h)"

    if feat == 'temp_delta_1h':
        hi = ref['q90'].get('temp_delta_1h', val); lo = ref['q10'].get('temp_delta_1h', val)
        if val >= hi: return "sharp rise in soil temperature (1h)"
        if val <= lo: return "sharp drop in soil temperature (1h)"
        return "unusual temperature change (1h)"

    if feat == 'ec_delta_1h':
        hi = ref['q90'].get('ec_delta_1h', val); lo = ref['q10'].get('ec_delta_1h', val)
        if val >= hi: return "sharp rise in soil EC (1h)"
        if val <= lo: return "sharp drop in soil EC (1h)"
        return "unusual EC change (1h)"

    # 6h slopes/trends (magnitude vs TRAIN 90th of |slope|)
    if feat == 'hum_slope_6h':
        thr = ref['q90_abs'].get('hum_slope_6h', np.nan)
        if pd.isna(thr) or thr == 0:
            thr = max(abs(ref['q90'].get('hum_slope_6h', 0.0)), abs(ref['q10'].get('hum_slope_6h', 0.0)))
        if abs(val) >= thr:
            return ("rising" if val > 0 else "falling") + " soil humidity (6h)"
        return "unusual humidity trend (6h)"

    if feat == 'temp_slope_6h':
        thr = ref['q90_abs'].get('temp_slope_6h', np.nan)
        if pd.isna(thr) or thr == 0:
            thr = max(abs(ref['q90'].get('temp_slope_6h', 0.0)), abs(ref['q10'].get('temp_slope_6h', 0.0)))
        if abs(val) >= thr:
            return ("rising" if val > 0 else "falling") + " soil temperature (6h)"
        return "unusual temperature trend (6h)"

    if feat == 'ec_slope_6h':
        thr = ref['q90_abs'].get('ec_slope_6h', np.nan)
        if pd.isna(thr) or thr == 0:
            thr = max(abs(ref['q90'].get('ec_slope_6h', 0.0)), abs(ref['q10'].get('ec_slope_6h', 0.0)))
        if abs(val) >= thr:
            return ("rising" if val > 0 else "falling") + " soil EC (6h)"
        return "unusual EC trend (6h)"

    # fallback
    return f"{PRETTY.get(feat, feat)} unusual"

def add_plain_language_reasons(d_line: pd.DataFrame, train_start: pd.Timestamp) -> pd.DataFrame:
    """
    Build a 'plain_english' column from top SHAP features (top1..top3) using TRAIN-window stats.
    Requires columns shap_top1..3 already present (or empty strings).
    """
    train_ref = _build_ref_stats(d_line[d_line['dt'] >= train_start], [c for c in BASIC_VARS if c in d_line.columns])
    out = d_line.copy()
    out['plain_english'] = ''
    anoms = out[out['is_anom']].copy()

    def _feat_name(s):
        return s.split(' (')[0] if isinstance(s, str) and s else ''

    phrases_all = []
    for _, r in anoms.iterrows():
        feats = [_feat_name(r.get('shap_top1','')), _feat_name(r.get('shap_top2','')), _feat_name(r.get('shap_top3',''))]
        phrases = [_phrase_for_feature(r, f, train_ref) for f in feats if f]
        # dedupe while preserving order
        seen = set()
        phrases = [p for p in phrases if not (p in seen or seen.add(p))]
        phrases_all.append(', '.join(phrases))

    out.loc[out['is_anom'], 'plain_english'] = phrases_all
    return out



In [6]:

# =======================================================
# "ground-truth" anomaly windows
# =======================================================
GT_WINDOWS = [
    # Known sensor manipulation days (per authors' email):
    # - 2023-07-12 (sensor repositioning)
    # - 2023-07-24 (sensor repositioning; stable after this date)
    # We give a conservative +/- 12 hours window each (you can narrow if desired).
    (pd.Timestamp('2023-07-12 00:00'), pd.Timestamp('2023-07-12 23:59'), 'repositioning'),
    (pd.Timestamp('2023-07-24 00:00'), pd.Timestamp('2023-07-24 23:59'), 'repositioning'),
    (pd.Timestamp('2023-06-29 18:10'), pd.Timestamp('2023-07-14 23:59'), 'unified watering as L100 until flowering'),

    # Optional: possible rain/temperature-drop event the user suspected on 2023-08-29.
    # If you verify it independently, you can include it; otherwise leave commented.
    # (pd.Timestamp('2023-08-29 00:00'), pd.Timestamp('2023-08-29 23:59'), 'weather_event'),
]


In [7]:

df = read_data(DATA_PATH)
df = add_watering_delta(df)
df, X_COLS = build_time_features(df)


df.head(3)


C:\Users\Ariel\AppData\Local\Temp\ipykernel_35152\1826661518.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['water_delta'] = df.groupby('line')['current_volume'].diff().fillna(0.0)
C:\Users\Ariel\AppData\Local\Temp\ipykernel_35152\1826661518.py:71: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df = df.groupby('line', group_keys=False).apply(_per_line)
C:\Users\Ariel\AppData\Local\Temp\ipykernel_35152\1826661518.py:71: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the ope

,dt,co2_env,humidity_env,temperature_env,pressure_env,current_volume,electrical_conductivity,humidity,temperature,line,...,temp_roll24h,ec_roll3h,ec_std3h,ec_roll24h,hum_delta_1h,temp_delta_1h,ec_delta_1h,hum_slope_6h,temp_slope_6h,ec_slope_6h
0,2023-06-29 18:10:00,449.0,36.5,33.9,1009.1,154.0,517.0,23.35,28.3,L100,...,28.30,517.0,0.000000,517.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-06-29 18:20:00,436.0,37.0,33.9,1009.1,154.0,519.0,23.35,28.4,L100,...,28.35,518.0,1.414214,518.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-06-29 18:30:00,434.0,35.0,33.9,1009.2,154.0,521.0,23.35,28.5,L100,...,28.40,519.0,2.000000,519.0,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:

line = DEFAULT_LINE  # change to 'L100'/'L60' as needed
d_line = df[df['line'] == line].reset_index(drop=True)

iforest, scaler, thr, scores_all = train_iforest(d_line, X_COLS, TRAIN_START)
d_line, periods = label_and_group_anomalies(d_line, scores_all, thr)




# attach GT labels (for hover + CSVs)
d_line  = assign_gt_labels_points(d_line, GT_WINDOWS)
periods = assign_gt_labels_periods(periods, GT_WINDOWS)

# optional SHAP reasons
if ENABLE_SHAP:
    d_line, periods = add_shap_reasons(iforest, scaler, d_line, periods, X_COLS, top_k=SHAP_TOP_K)

# Plain-English reasons (from TRAIN stats + top SHAP features)
d_line = add_plain_language_reasons(d_line, TRAIN_START)



print(f'Line: {line} | Train start: {TRAIN_START} | Score threshold (q={ANOMALY_SCORE_QUANTILE:.3f}): {thr:.4f}')
print('Predicted periods:', len(periods))
periods.head()


0
NaNs in X_train: 0
NaNs in X_all: 126
Line: L30 | Train start: 2023-07-25 00:00:00 | Score threshold (q=0.010): -0.1661
Predicted periods: 38


,start,end,n_points,gt_window_label,shap_period_summary
0,2023-06-29 19:10:00,2023-06-30 03:00:00,47,unified watering as L100 until flowering,"humidity (~1.26), hum_roll3h (~1.24), ec_roll3..."
1,2023-06-30 03:50:00,2023-06-30 03:50:00,1,unified watering as L100 until flowering,"humidity (~1.32), hum_roll3h (~1.29), electric..."
2,2023-06-30 09:00:00,2023-06-30 11:00:00,13,unified watering as L100 until flowering,"hum_roll3h (~1.26), humidity (~1.25), ec_roll3..."
3,2023-06-30 16:20:00,2023-06-30 23:50:00,46,unified watering as L100 until flowering,"humidity (~1.21), hum_roll3h (~1.12), electric..."
4,2023-07-01 11:40:00,2023-07-02 04:30:00,99,unified watering as L100 until flowering,"hum_roll3h (~1.18), humidity (~1.18), electric..."


# Plotting Humidity with Anomalies

In [10]:
def plot_interactive(d_line, periods, gt_windows, title=''):
    d = d_line.copy()
    shapes = []

    # --- GT windows (bright yellow) ---
    for s, e, _label in gt_windows:
        shapes.append(dict(
            type='rect', x0=s, x1=e, y0=0, y1=1, xref='x', yref='paper',
            fillcolor='yellow', opacity=0.1, line=dict(width=0), layer='below'
        ))

    # --- Watering periods (as shaded bands, not markers) ---
    wper = periods_from_flag(d, flag_col='watering', time_col='dt')
    for row in wper.itertuples(index=False):
        shapes.append(dict(
            type='rect',
            x0=row.start,
            x1=row.end + pd.Timedelta(minutes=10),  # include the last step
            y0=0, y1=1, xref='x', yref='paper',
            fillcolor="#049df0", opacity=0.1, line=dict(width=0), layer='below'
        ))

    # --- Anomaly periods (red) ---
    for row in periods.itertuples(index=False):
        shapes.append(dict(
            type='rect',
            x0=row.start,
            x1=row.end + pd.Timedelta(minutes=10),
            y0=0, y1=1, xref='x', yref='paper',
            fillcolor='red', opacity=0.2, line=dict(width=0), layer='below'
        ))

    # Figure & traces
    fig = go.Figure()

    # main signal
    fig.add_trace(go.Scatter(
        x=d['dt'], y=d['humidity'],
        mode='lines', name='Soil humidity'
    ))

    # anomaly points with rich hover (GT + plain-English reasons)
    da = d[d['is_anom']].copy()
    hover_text = []
    for _, r in da.iterrows():
        reasons = r.get('plain_english') or r.get('shap_summary') or ''
        ht = (f"<b>{r['dt']}</b><br>"
              f"humidity: {r['humidity']:.2f}%<br>"
              f"score: {r['score']:.3f}<br>"
              f"GT: <b>{r['gt_window_label'] or '—'}</b>")
        if reasons:
            ht += f"<br>why: {reasons}"
        hover_text.append(ht)

    fig.add_trace(go.Scatter(
        x=da['dt'], y=da['humidity'],
        mode='markers', name='Anomaly',
        marker=dict(size=7, symbol='circle-open'),
        hovertemplate='%{text}', text=hover_text
    ))

    # add legend entries for shaded bands using small, non-intrusive markers
    # GT window (yellow)
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=10, color='yellow',opacity = 0.22, symbol='square'),
        name='GT window',
        hoverinfo='skip',
        showlegend=True
    ))
    # Watering (blue)
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=10, color='cornflowerblue', opacity=0.12, symbol='square'),
        name='Watering',
        hoverinfo='skip',
        showlegend=True
    ))
    # Anomaly periods (red)
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=10, color='red', opacity=0.15, symbol='square'),
        name='Anomaly period',
        hoverinfo='skip',
        showlegend=True
    ))




    # apply shaded bands (behind traces)
    fig.update_layout(
        shapes=shapes,
        title=title,
        legend=dict(orientation='h'),
        template='plotly_white',
        hovermode="x unified"
    )
    fig.update_xaxes(title='Time')
    fig.update_yaxes(title='%RH (soil)')
    fig.show()


plot_interactive(d_line, periods, GT_WINDOWS, title=f"{line} — humidity with anomalies, GT & watering")


# Plotting all soil metric with Anomalies

In [16]:
def plot_interactive_multiplot(d_line, periods, gt_windows, thr, title=''):
    from plotly.subplots import make_subplots
    # ---------- prep ----------
    d = d_line.copy().reset_index(drop=True)
    df_plot = d.set_index('dt').copy()

    # watering per-step (m³ per 10-min step) & rolling doses
    step_col = 'water_delta'  # <- from add_watering_delta()
    df_plot['water_delta_m3_10m'] = df_plot[step_col].fillna(0.0)

    w6h  = int(6*60 / TIMESTEP_MIN)
    w24h = int(24*60 / TIMESTEP_MIN)
    df_plot['dose_6h_m3']  = df_plot['water_delta_m3_10m'].rolling(w6h,  min_periods=1).sum()
    df_plot['dose_24h_m3'] = df_plot['water_delta_m3_10m'].rolling(w24h, min_periods=1).sum()

    # IF score for plotting
    df_plot['if_score'] = df_plot['score']

    # watering periods as [start, end]
    irrigation_periods = periods_from_flag(d, flag_col='watering', time_col='dt')

    # anomaly periods with optional "peak" (most negative score inside each period)
    anomaly_periods = periods.copy()
    if not anomaly_periods.empty:
        peak_times, peak_scores = [], []
        for row in anomaly_periods.itertuples(index=False):
            win = df_plot.loc[(df_plot.index >= row.start) & (df_plot.index <= row.end), 'if_score']
            if len(win):
                pt = win.idxmin()  # most negative (strongest anomaly)
                peak_times.append(pt)
                peak_scores.append(float(win.loc[pt]))
            else:
                peak_times.append(pd.NaT)
                peak_scores.append(np.nan)
        anomaly_periods = anomaly_periods.assign(peak_time=peak_times, peak_score=peak_scores)

    # ---------- figure ----------
    fig = make_subplots(
        rows=5, cols=1, shared_xaxes=True,
        row_heights=[0.24, 0.19, 0.19, 0.19, 0.19],
        vertical_spacing=0.04,
        subplot_titles=(
            "Soil Humidity",
            "Soil Temperature",
            "Electrical Conductivity (EC)",
            "Watering (per-step & rolling doses)",
            "Isolation Forest Score (threshold)"
        )
    )

    # 1) humidity
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['humidity'],
                             mode='lines', name='humidity'),
                  row=1, col=1)
    
    # ---- anomaly markers with rich hover on the humidity panel ----
    # choose the best "reasons" column available
    reason_col = 'human_reasons' if 'human_reasons' in d.columns else (
        'plain_english' if 'plain_english' in d.columns else (
            'shap_summary' if 'shap_summary' in d.columns else None
        )
    )

    da = d[d['is_anom']].copy()
    hover_text = []
    for _, r in da.iterrows():
        reasons = (r.get(reason_col) if reason_col else '') or ''
        ht = (f"<b>{r['dt']}</b><br>"
              f"humidity: {r['humidity']:.2f}%<br>"
              f"score: {r['score']:.3f}<br>"
              f"GT: <b>{r.get('gt_window_label') or '—'}</b>")
        if reasons:
            ht += f"<br>why: {reasons}"
        hover_text.append(ht)

    fig.add_trace(
        go.Scatter(
            x=da['dt'], y=da['humidity'],
            mode='markers',
            name='Anomaly',
            marker=dict(size=7, symbol='circle-open'),
            hovertemplate='%{text}', text=hover_text
        ),
        row=1, col=1
    )


    # 2) temperature
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['temperature'],
                             mode='lines', name='temperature'),
                  row=2, col=1)

    # 3) EC
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['electrical_conductivity'],
                             mode='lines', name='EC'),
                  row=3, col=1)

    # 4) watering — bars + rolling doses
    fig.add_trace(go.Bar(x=df_plot.index, y=df_plot['water_delta_m3_10m'],
                         name='water_delta (m³/step)', opacity=0.6),
                  row=4, col=1)
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['dose_6h_m3'],
                             mode='lines', name='dose_6h (m³)', line=dict(dash='dot')),
                  row=4, col=1)
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['dose_24h_m3'],
                             mode='lines', name='dose_24h (m³)', line=dict(dash='dash')),
                  row=4, col=1)

    # 5) IF score
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['if_score'],
                             mode='lines', name='IF score'),
                  row=5, col=1)
    # threshold line on row 5
    try:
        fig.add_hline(y=thr, line_dash='dash', row=5, col=1)
    except Exception:
        # fallback if add_hline is unavailable; draw a shape on yaxis5
        fig.add_shape(type='line', x0=df_plot.index.min(), x1=df_plot.index.max(),
                      y0=thr, y1=thr, xref='x5', yref='y5',
                      line=dict(dash='dash'))

    # ---------- shaded bands across ALL rows ----------
    PAD = pd.Timedelta(0)

    # GT windows (bright yellow)
    for (s, e, _lab) in gt_windows:
        fig.add_vrect(x0=s-PAD, x1=e+PAD, fillcolor="yellow",
                      opacity=0.22, line_width=0, layer="below",
                      row="all", col=1)

    # watering periods (soft blue)
    for r in irrigation_periods.itertuples(index=False):
        fig.add_vrect(x0=r.start-PAD, x1=r.end+PAD, fillcolor="cornflowerblue",
                      opacity=0.12, line_width=0, layer="below",
                      row="all", col=1)

    # anomaly clusters (red)
    for r in anomaly_periods.itertuples(index=False):
        fig.add_vrect(x0=r.start-PAD, x1=r.end+PAD, fillcolor="red",
                      opacity=0.15, line_width=0, layer="below",
                      row="all", col=1)





    # ---------- legend entries for shaded periods ----------
    # Add invisible traces to create legend entries for the shaded bands
    # Place them in row 1 so they don't interfere with other subplots
    
    # GT window legend (yellow)
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color='yellow',opacity = 0.22, symbol='square', line=dict(color='black', width=1)),
        name='GT window',
        hoverinfo='skip',
        showlegend=True
    ), row=1, col=1)
    
    # Watering period legend (blue)
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color='cornflowerblue', opacity=0.12, symbol='square', line=dict(color='black', width=1)),
        name='Watering period',
        hoverinfo='skip',
        showlegend=True
    ), row=1, col=1)
    
    # Anomaly period legend (red)
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color='red', opacity=0.15, symbol='square', line=dict(color='black', width=1)),
        name='Anomaly period',
        hoverinfo='skip',
        showlegend=True
    ), row=1, col=1)





    # layout
    fig.update_layout(
        height=1100,
        title=title,
        legend=dict(orientation='h', y=1.04, x=0.0),
        margin=dict(l=60, r=30, t=70, b=40),
        barmode='overlay',
        template='plotly_white',
        hovermode="x unified"

        
    )
    fig.update_xaxes(showspikes=True, spikemode='across', spikesnap='cursor')
    fig.update_yaxes(matches=None)  # keep independent y ranges
    os.makedirs('docs', exist_ok=True)
    fig.write_html(f"docs/soil_humidity_{line}.html", include_plotlyjs=True, full_html=True)
    fig.show()


In [17]:
plot_interactive_multiplot(
    d_line, periods, GT_WINDOWS, thr,
    title=f"{line} — Sensors, Watering & Anomaly Clusters"
)

# Precision Results

In [13]:

metrics = precision_against_windows(periods, d_line[['dt','is_anom']].copy(), GT_WINDOWS)
print(f"Precision (point-wise): {metrics['point_precision']:.4f}")
print(f"Precision (period-wise): {metrics['period_precision']:.4f}")



Precision (point-wise): 0.9486
Precision (period-wise): 0.8421


# Exporting to csv

In [10]:
from pathlib import Path
# create output directory inside current working directory (not C:\)
out_dir = Path.cwd() / 'data' / 'iforest_outputs_plus'
out_dir.mkdir(parents=True, exist_ok=True)

# Points CSV
cols_points = ['line','dt','humidity','score','is_anom','gt_window_label']
extra_cols  = [c for c in ['plain_english','shap_top1','shap_top2','shap_top3','shap_summary'] if c in d_line.columns]
d_line[cols_points + extra_cols].to_csv(out_dir / f'pred_points_{line}.csv', index=False)


# Periods CSV
cols_periods = ['start','end','n_points','gt_window_label']
if 'shap_period_summary' in periods.columns:
    cols_periods.append('shap_period_summary')
periods[cols_periods].to_csv(out_dir / f'pred_periods_{line}.csv', index=False)

print("Saved:", out_dir)


Saved: c:\Users\Ariel\Studies\Year C\2nd Semester\Urban agriculture\IoT-based Dataset of a Tomato Cultivation Under Different Irrigation Regimes\data\iforest_outputs_plus
